In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.csv('/Volumes/databrickstutorial/default/data/BigMart Sales.csv', header='true', inferSchema='true')
json_df = spark.read.json('/Volumes/databrickstutorial/default/data/drivers.json')

In [0]:
df.createOrReplaceTempView("temp_df")
spark.sql("SELECT spark_partition_id() FROM temp_df").distinct().count()

### Changing DEFAULT Patition Size to 128KB

In [0]:
spark.conf.set('spark.sql.files.maxPartitionBytes',131072)

In [0]:
df.createOrReplaceTempView("temp_df")
spark.sql("SELECT spark_partition_id() FROM temp_df").distinct().count()

### Changing DEFAULT Patition Size to 128MB

In [0]:
spark.conf.set('spark.sql.files.maxPartitionBytes',134217728)

#### Repartitioning

In [0]:
df = df.repartition(10)

In [0]:
df.createOrReplaceTempView("temp_df")
spark.sql("SELECT spark_partition_id() FROM temp_df").distinct().count()

In [0]:
df.display()

#### Get Partition Info

In [0]:
df.withColumn('partition_id', spark_partition_id()).display()

#### Data Writing

In [0]:
df.write.format('parquet').mode('append').option('path','/Volumes/databrickstutorial/default/data/parquetWrite').save()

#### New Data Reading

In [0]:
df_new = spark.read.format('parquet').load('/Volumes/databrickstutorial/default/data/parquetWrite')
df_new = df_new.filter(df_new['Outlet_Location_Type'] == 'Tier 1')

In [0]:
df_new.display()

## SCANNING OPTIMIZATION

In [0]:
df.write.format('parquet').mode('append').partitionBy('Outlet_Location_Type').option('path','/Volumes/databrickstutorial/default/data/parquetWriteOpt')\
    .save()

In [0]:
df_new = spark.read.format('parquet').load('/Volumes/databrickstutorial/default/data/parquetWriteOpt')
df_new = df_new.filter(df_new['Outlet_Location_Type'] == 'Tier 3')
df_new.display()

#JOINS OPTIMIZATION